In [ ]:
#| export machine_learning.note_linking
from datasets import Dataset
import random
from typing import Literal, Optional, TypedDict


from trouver.machine_learning.note_data import (
    NoteLinkEnum, InfoNoteData, NotatNoteData)


In [ ]:
from trouver.machine_learning.note_linking import NotePairData, link_types_for_note_pair_data, sieve_note_data_pairs

In [ ]:
from unittest.mock import MagicMock

## Converting a note pair into a string and data augmentation 

**Preparing Inputs for NLP Models**

Once note pairs are selected, they must be converted into a single text string suitable for transformer models (BERT, T5). This involves:

    Formatting: Concatenating the "Origin" and "Relied" note data with a specific separator token ([SEP] for BERT, </s> for T5).

    Augmentation: To make the model robust, we randomly perturb the text (e.g. removing LaTeX commands, making bad formatting) and occasionally erase positional metadata (section numbers, etc.) to prevent overfitting to specific document structures.


In [ ]:
#| export machine_learning.note_linking
def _erase_position_metadata(
        augmentation: Literal['high', 'mid', 'low'] | None
        ) -> bool:
    """
    Randomly determines whether to erase positional metadata based on augmentation intensity.
    
    - 'high': 30% chance
    - 'mid':  20% chance
    - 'low':  10% chance
    """
    if augmentation is None: return False
    
    rand_value = random.random()
    thresholds = {'high': 0.3, 'mid': 0.2, 'low': 0.1}
    return rand_value < thresholds.get(augmentation, 0.0)


In [ ]:
#| export machine_learning.note_linking
def string_from_note_pair(
        pair_data: NotePairData, # The pair of notes to convert.
        format: Literal['bert', 't5'] # The model architecture determines the separator token.
        ) -> str: # A single string combining both notes.
    """
    Formats a pair of notes into a single input string for NLP models.
    
    Combines the `data_string` of both notes, separated by a model-specific delimiter.
    """
    origin_data = pair_data['origin_note']
    relied_data = pair_data['relied_note']
    origin_data_string = origin_data.data_string(format)
    relied_data_string = relied_data.data_string(format)
    
    if format == 'bert':
        return f'{origin_data_string}\n\n[SEP]\n\n{relied_data_string}'
    else:
        # T5 typically uses </s> or specific sentinel tokens depending on pre-training
        return f'{origin_data_string}\n\n</s>\n\n{relied_data_string}'


In [ ]:
#| export machine_learning.note_linking
def augment_note_pair(
        pair_data: NotePairData, # The original note pair.
        augmentation: Optional[Literal['high', 'mid', 'low']] = None, # Intensity of augmentation.
        include_position_data_for_origin: bool = True, # Force inclusion/exclusion of origin metadata.
        include_position_data_for_relied: bool = True # Force inclusion/exclusion of relied metadata.
        ) -> NotePairData: # A new, modified NotePairData object.
    """
    Creates an augmented copy of a note pair for training data variety.
    
    Applies text perturbations (via `randomly_modify`) and optionally erases 
    positional metadata (section numbers, etc.) to force the model to focus on content.
    """
    origin_data = pair_data['origin_note'].deepcopy()
    relied_data = pair_data['relied_note'].deepcopy()
    
    # Determine if we should erase metadata (probabilistic OR forced)
    erase_origin = _erase_position_metadata(augmentation) or not include_position_data_for_origin
    erase_relied = _erase_position_metadata(augmentation) or not include_position_data_for_relied
    
    if augmentation is not None:
        origin_data.randomly_modify(augmentation, erase_position_metadata=erase_origin)
        relied_data.randomly_modify(augmentation, erase_position_metadata=erase_relied)
        
    return NotePairData(origin_note=origin_data, relied_note=relied_data)

In [ ]:
from unittest.mock import MagicMock, patch

def test_string_formatting():
    # Setup Mocks
    origin = MagicMock(); origin.data_string.return_value = "ORIGIN_TEXT"
    relied = MagicMock(); relied.data_string.return_value = "RELIED_TEXT"
    pair = {"origin_note": origin, "relied_note": relied}
    
    # BERT format
    assert string_from_note_pair(pair, "bert") == "ORIGIN_TEXT\n\n[SEP]\n\nRELIED_TEXT"
    
    # T5 format
    assert string_from_note_pair(pair, "t5") == "ORIGIN_TEXT\n\n</s>\n\nRELIED_TEXT"

@patch('random.random')
def test_metadata_erasure_probability(mock_rand):
    # Test 'high' probability (0.3)
    mock_rand.return_value = 0.25 # Below 0.3
    assert _erase_position_metadata('high') is True
    
    mock_rand.return_value = 0.35 # Above 0.3
    assert _erase_position_metadata('high') is False
    
    # Test None
    assert _erase_position_metadata(None) is False

def test_augmentation_logic():
    # Setup recursive mocks for deepcopy/randomly_modify
    origin = MagicMock()
    origin.deepcopy.return_value = origin # Simplify for test
    relied = MagicMock()
    relied.deepcopy.return_value = relied
    
    pair = {"origin_note": origin, "relied_note": relied}
    
    # Run Augmentation
    with patch('__main__._erase_position_metadata', return_value=True):
        augment_note_pair(pair, 'mid')
        
    # Verify modification calls
    origin.randomly_modify.assert_called_with('mid', erase_position_metadata=True)
    relied.randomly_modify.assert_called_with('mid', erase_position_metadata=True)

test_string_formatting()
test_metadata_erasure_probability()
test_augmentation_logic()

**Constructing the Final Dataset**

We compile the processed pairs into a standard format (`NoteLinkingDataPoint`) compatible with HuggingFace Dataset objects. This format includes:

1. Input Text: The augmented, concatenated string.
2. Labels: A list of NoteLinkEnum names representing the valid relationships between the notes (e.g., ['INFO_TO_INFO_IN_CONTENT', 'INFO_TO_INFO_IN_SEE_ALSO']).

The `dataset_from_note_data` function orchestrates the entire pipeline: sieving pairs, converting them to data points, applying augmentations, and returning a ready-to-train Dataset.

In [ ]:
#| export machine_learning.note_linking
class NoteLinkingDataPoint(TypedDict):
    """
    A dictionary structure representing a single training example for the model.
    """
    origin_note_name: str # Name of the source note.
    relied_note_name: str # Name of the target note.
    input_text: str # Concatenated text of both notes (augmented).
    link_types: list[str] # List of active link types (labels) for multi-label classification.

In [ ]:
#| export machine_learning.note_linking
def dict_data_point_from_pair(
        pair_data: NotePairData, # The note pair to convert.
        format: Literal['bert', 't5'] # Format for the input text string.
        ) -> NoteLinkingDataPoint: # The structured training example.
    """
    Converts a raw `NotePairData` into a labeled `NoteLinkingDataPoint` for training.
    
    Extracts the link types (labels) and generates the formatted input text.
    """
    origin_note_name = pair_data['origin_note'].note_name
    relied_note_name = pair_data['relied_note'].note_name
    input_text = string_from_note_pair(pair_data, format)    
    
    # Get active links as a set of Enums, convert to list of strings for JSON/Dataset compatibility
    active_links = link_types_for_note_pair_data(pair_data)
    link_types_str = [link.name for link in active_links]
    
    return NoteLinkingDataPoint(
        origin_note_name=origin_note_name,
        relied_note_name=relied_note_name,
        input_text=input_text,
        link_types=link_types_str
    )

In [ ]:
# Mock data (using the helper from previous sections)

def mock_note_data(name, reverse_count=0, direct_links=None, tags=None, notation_str=""):
    """
    Creates a mock NoteData object.
    
    Args:
        direct_links: Dictionary mapping NoteName -> Set of NoteLinkEnum
                      Example: {"TargetNote": {NoteLinkEnum.INFO_TO_INFO_IN_CONTENT}}
    """
    m = MagicMock()
    m.note_name = name
    
    # Mock data_string to prevent string_from_note_pair failure
    m.data_string.return_value = f"Mock Data for {name}"
    
    # Correctly mocking deepcopy to return itself (or a new mock) for augmentation tests
    m.deepcopy.return_value = m 

    m.reverse_linked_notes = {f"in_{i}" for i in range(reverse_count)}
    
    # CRITICAL FIX: Ensure direct_links values are Sets/Iterables, not Integers
    m.directly_linked_notes = direct_links if direct_links else {}
    
    m.tags = tags
    m.parsed = MagicMock()
    m.parsed.notation_str = notation_str if notation_str else name
    return m

#| example
# Setup mock data with CORRECT Enum usage (with underscores) and structure (Set, not Int)
origin_note = mock_note_data(
    "NoteA", 
    direct_links={"NoteB": {NoteLinkEnum.INFO_TO_INFO_IN_CONTENT}} # Value must be a SET
)
relied_note = mock_note_data("NoteB")
pair = {"origin_note": origin_note, "relied_note": relied_note}

# Convert single pair
data_point = dict_data_point_from_pair(pair, format='bert')
print(f"Input Text: {data_point['input_text'][:40]}...")
print(f"Labels: {data_point['link_types']}")

Input Text: Mock Data for NoteA

[SEP]

Mock Data fo...
Labels: ['INFO_TO_INFO_IN_CONTENT']


In [ ]:
# #| export machine_learning.note_linking
# def dataset_from_note_data(
#         info_note_data: dict[str, InfoNoteData], # Pool of information notes.
#         notat_note_data: dict[str, NotatNoteData], # Pool of notation notes.
#         augment: bool, # Whether to generate augmented copies of data points.
#         format: Literal['bert', 't5'] # Model format for text encoding.
#         ) -> Dataset: # A HuggingFace Dataset ready for training.
#     """
#     Full pipeline: Sieves note pairs, augments them, and compiles a HuggingFace Dataset.
    
#     If `augment` is True, generates 3 additional versions (low, mid, high intensity) 
#     for every sampled pair, effectively quadrupling the dataset size.
#     """
#     # 1. Gather raw pairs via heuristic sieving
#     note_data_pairs: list[NotePairData] = sieve_note_data_pairs(
#         info_note_data, notat_note_data)
    
#     dict_data: list[NoteLinkingDataPoint] = []
    
#     for pair_data in note_data_pairs:
#         # Add original (un-augmented) data point
#         dict_data.append(dict_data_point_from_pair(
#             pair_data, format))
        
#         if not augment:
#             continue
            
#         # Add 3 augmented versions
#         for augmentation in ['low', 'mid', 'high']:
#             augmented_pair_data = augment_note_pair(
#                 pair_data, augmentation)
#             dict_data.append(dict_data_point_from_pair(
#                 augmented_pair_data, format))
                
#     return Dataset.from_list(dict_data)

In [ ]:
#| export machine_learning.note_linking
def dataset_from_note_data(
        info_note_data: dict[str, InfoNoteData],
        notat_note_data: dict[str, NotatNoteData],
        augment: bool,
        format: Literal['bert', 't5'],
        batch_size: int = 32, # NEW: Process pairs in batches during augmentation
        ) -> Dataset:
    """
    Full pipeline: Sieves note pairs, augments them, and compiles a HuggingFace Dataset.
    
    If `augment` is True, generates 3 additional versions (low, mid, high intensity) 
    for every sampled pair, effectively quadrupling the dataset size.
    
    Args:
        batch_size: Number of pairs to process simultaneously (default 32)
    """
    note_data_pairs: list[NotePairData] = sieve_note_data_pairs(
        info_note_data, notat_note_data)
    
    dict_data: list[NoteLinkingDataPoint] = []
    
    # Process in batches for efficiency
    for i in range(0, len(note_data_pairs), batch_size):
        batch_pairs = note_data_pairs[i:i+batch_size]
        
        for pair_data in batch_pairs:
            dict_data.append(dict_data_point_from_pair(pair_data, format))
            
            if augment:
                for augmentation in ['low', 'mid', 'high']:
                    augmented_pair_data = augment_note_pair(
                        pair_data, augmentation)
                    dict_data.append(dict_data_point_from_pair(
                        augmented_pair_data, format))
    
    return Dataset.from_list(dict_data)


In [ ]:
#| test
@patch('__main__.sieve_note_data_pairs')
@patch('__main__.augment_note_pair')
def test_dataset_generation_loop(mock_augment, mock_sieve):
    # --- 1. Setup Mock Objects that return REAL STRINGS ---
    
    # Mock Origin/Relied notes for the initial Sieve output
    sieve_origin = MagicMock()
    sieve_origin.note_name = "OriginNote"   # <--- MUST be a string
    sieve_origin.data_string.return_value = "Origin Data"
    sieve_origin.directly_linked_notes = {}  # Empty dict needed for link_types_for_note_pair_data
    
    sieve_relied = MagicMock()
    sieve_relied.note_name = "ReliedNote"   # <--- MUST be a string
    sieve_relied.data_string.return_value = "Relied Data"
    
    # The sieve returns this pair
    mock_sieve.return_value = [{
        "origin_note": sieve_origin, 
        "relied_note": sieve_relied
    }]

    # --- 2. Setup Mock Objects for Augmentation Output ---
    
    aug_origin = MagicMock()
    aug_origin.note_name = "AugOriginNote" # <--- MUST be a string
    aug_origin.data_string.return_value = "Aug Data"
    aug_origin.directly_linked_notes = {}
    
    aug_relied = MagicMock()
    aug_relied.note_name = "AugReliedNote" # <--- MUST be a string
    aug_relied.data_string.return_value = "Aug Data"
    
    # augment_note_pair returns this pair
    mock_augment.return_value = {
        "origin_note": aug_origin, 
        "relied_note": aug_relied
    }
    
    # --- Run Tests ---
    
    # 1. No Augmentation
    ds = dataset_from_note_data({}, {}, augment=False, format='bert')
    assert len(ds) == 1
    # Verify the dataset actually contains the string "OriginNote"
    assert ds[0]['origin_note_name'] == "OriginNote"
    
    # 2. With Augmentation
    ds_aug = dataset_from_note_data({}, {}, augment=True, format='bert')
    assert len(ds_aug) == 4
    # The first item is original
    assert ds_aug[0]['origin_note_name'] == "OriginNote"
    # The subsequent items are augmented (check name from mocked augmented output)
    assert ds_aug[1]['origin_note_name'] == "AugOriginNote"

test_dataset_generation_loop()